In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
p = pathlib.Path.cwd()
for q in (p, *p.parents):
    s = q / "src" / "ftbp"   # <- change "ftbp" if you rename the package
    if s.exists():
        sys.path.insert(0, str(s.parent))  # add .../src
        break
else:
    raise RuntimeError("src/ftbp not found")

In [ ]:
import numpy as np
from scipy.stats import norm, cauchy, uniform
import pandas as pd
from ftbp.estimators import *
from ftbp.wald import *

In [ ]:
### the gap experiment
from tqdm import tqdm
n = 1000
ms = [20 + i * 40 for i in range(13)]
# ns = [100]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm, cauchy, uniform]
loss_type = 'huber'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479

all_results = []
for m in ms:
    for dist in dists:
        for seed in tqdm(seeds):
            x = dist.rvs(size=n)
            theta_hat = estimate_theta(x, delta=delta, loss_type=loss_type)
            eta = max(eta_theta_plus(x, theta_hat, m, delta=delta, loss_type=loss_type), eta_theta_minus(x, theta_hat, m, delta=delta, loss_type=loss_type))

            all_results.append({
                'm': m,
                'dist': getattr(dist, 'name', dist.__class__.__name__),
                'seed': seed,
                'eta': eta
            })

df = pd.DataFrame(all_results)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# 1) pick your style & context
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

# thresholded manually before 480
df = df[df['m'] < 480]

# 3) build the scatter + regression line
plt.figure(figsize=(8,6))
sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'norm'],
    color=palette[0],
    label=f'Normal',
    marker='o',
    linestyle='-',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'cauchy'],
    color=palette[1],
    label=f'Cauchy',
    marker='s',
    linestyle='--',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'uniform'],
    color=palette[2],
    label=f'Uniform',
    marker='^',
    linestyle=':',
    alpha=0.8
)

# 4) polish labels & legend
plt.xlabel("$m$")
plt.ylabel("$\eta$")
# plt.title("Scaling of BP_reject bound gap with $n$ (normal)")
plt.legend(loc='upper left', ncols=1, frameon=False)
plt.tight_layout()
plt.savefig("eta.pdf", bbox_inches='tight')


In [ ]:
### the gap experiment
from tqdm import tqdm
ns = [500, 1000, 2000]
ms = [20/1000 + i * 40/1000 for i in range(13)]
# ns = [100]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm]
loss_type = 'huber'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479

all_results = []
for n in ns:
    for m in ms:
        for dist in dists:
            for seed in tqdm(seeds):
                x = dist.rvs(size=n)
                theta_hat = estimate_theta(x, delta=delta, loss_type=loss_type)
                eta = max(eta_theta_plus(x, theta_hat, int(m*n), delta=delta, loss_type=loss_type), eta_theta_minus(x, theta_hat, int(m*n), delta=delta, loss_type=loss_type))

                all_results.append({
                    'm': m,
                    # 'dist': getattr(dist, 'name', dist.__class__.__name__),
                    'seed': seed,
                    'eta': eta,
                    'n': n,
                })

df = pd.DataFrame(all_results)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# 1) pick your style & context
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

# thresholded manually before 450
df = df[df['m'] < 500/1000]

# 3) build the scatter + regression line
plt.figure(figsize=(8,6))
sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 500],
    color=palette[0],
    label=f'500',
    marker='o',
    linestyle='-',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 1000],
    color=palette[1],
    label=f'1000',
    marker='s',
    linestyle='--',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 2000],
    color=palette[2],
    label=f'2000',
    marker='^',
    linestyle=':',
    alpha=0.8
)

# 4) polish labels & legend
plt.xlabel("$m/n$")
plt.ylabel("$\eta$")
# plt.title("Scaling of BP_reject bound gap with $n$ (normal)")
plt.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=3, frameon=False)
plt.tight_layout()
plt.savefig("eta_n.pdf", bbox_inches='tight')

# r2      = model.rsquared
# r2_adj  = model.rsquared_adj
# print(f"R² = {r2:.3f},  Adjusted R² = {r2_adj:.3f}")
# print("F-statistic =", model.fvalue)
# print("F p‑value   =", model.f_pvalue)


In [ ]:
### the gap experiment
from tqdm import tqdm
n = 1000
ms = [20 + i * 40 for i in range(13)]
# ns = [100]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm, cauchy, uniform]
loss_type = 'logcosh'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479

all_results = []
for m in ms:
    for dist in dists:
        for seed in tqdm(seeds):
            x = dist.rvs(size=n)
            theta_hat = estimate_theta(x, delta=delta, loss_type=loss_type)
            eta = max(eta_theta_plus(x, theta_hat, m, delta=delta, loss_type=loss_type), eta_theta_minus(x, theta_hat, m, delta=delta, loss_type=loss_type))

            all_results.append({
                'm': m,
                'dist': getattr(dist, 'name', dist.__class__.__name__),
                'seed': seed,
                'eta': eta
            })

df = pd.DataFrame(all_results)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# 1) pick your style & context
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

# thresholded manually before 450
df = df[df['m'] < 480]

# 3) build the scatter + regression line
plt.figure(figsize=(8,6))
sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'norm'],
    color=palette[0],
    label=f'Normal',
    marker='o',
    linestyle='-',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'cauchy'],
    color=palette[1],
    label=f'Cauchy',
    marker='s',
    linestyle='--',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'uniform'],
    color=palette[2],
    label=f'Uniform',
    marker='^',
    linestyle=':',
    alpha=0.8
)

# 4) polish labels & legend
plt.xlabel("$m$")
plt.ylabel("$\eta$")
# plt.title("Scaling of BP_reject bound gap with $n$ (normal)")
plt.legend(loc='upper left', ncols=1, frameon=False)
plt.tight_layout()
plt.savefig("eta_logcosh.pdf", bbox_inches='tight')


In [ ]:
### the gap experiment
from tqdm import tqdm
ns = [500, 1000, 2000]
ms = [20/1000 + i * 40/1000 for i in range(13)]
# ns = [100]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm]
loss_type = 'logcosh'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479

all_results = []
for n in ns:
    for m in ms:
        for dist in dists:
            for seed in tqdm(seeds):
                x = dist.rvs(size=n)
                theta_hat = estimate_theta(x, delta=delta, loss_type=loss_type)
                eta = max(eta_theta_plus(x, theta_hat, int(m*n), delta=delta, loss_type=loss_type), eta_theta_minus(x, theta_hat, int(m*n), delta=delta, loss_type=loss_type))

                all_results.append({
                    'm': m,
                    # 'dist': getattr(dist, 'name', dist.__class__.__name__),
                    'seed': seed,
                    'eta': eta,
                    'n': n,
                })

df = pd.DataFrame(all_results)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# 1) pick your style & context
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

# thresholded manually before 450
df = df[df['m'] < 500/1000]

# 3) build the scatter + regression line
plt.figure(figsize=(8,6))
sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 500],
    color=palette[0],
    label=f'500',
    marker='o',
    linestyle='-',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 1000],
    color=palette[1],
    label=f'1000',
    marker='s',
    linestyle='--',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 2000],
    color=palette[2],
    label=f'2000',
    marker='^',
    linestyle=':',
    alpha=0.8
)

# 4) polish labels & legend
plt.xlabel("$m/n$")
plt.ylabel("$\eta$")
# plt.title("Scaling of BP_reject bound gap with $n$ (normal)")
plt.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=3, frameon=False)
plt.tight_layout()
plt.savefig("eta_n_logcosh.pdf", bbox_inches='tight')

In [ ]:
### the gap experiment
from tqdm import tqdm
n = 1000
ms = [20 + i * 40 for i in range(13)]
# ns = [100]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm, cauchy, uniform]
loss_type = 'concordant'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479

all_results = []
for m in ms:
    for dist in dists:
        for seed in tqdm(seeds):
            x = dist.rvs(size=n)
            theta_hat = estimate_theta(x, delta=delta, loss_type=loss_type)
            eta = max(eta_theta_plus(x, theta_hat, m, delta=delta, loss_type=loss_type), eta_theta_minus(x, theta_hat, m, delta=delta, loss_type=loss_type))

            all_results.append({
                'm': m,
                'dist': getattr(dist, 'name', dist.__class__.__name__),
                'seed': seed,
                'eta': eta
            })

df = pd.DataFrame(all_results)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# 1) pick your style & context
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

# thresholded manually before 450
df = df[df['m'] < 480]

# 3) build the scatter + regression line
plt.figure(figsize=(8,6))
sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'norm'],
    color=palette[0],
    label=f'Normal',
    marker='o',
    linestyle='-',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'cauchy'],
    color=palette[1],
    label=f'Cauchy',
    marker='s',
    linestyle='--',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['dist'] == 'uniform'],
    color=palette[2],
    label=f'Uniform',
    marker='^',
    linestyle=':',
    alpha=0.8
)

# 4) polish labels & legend
plt.xlabel("$m$")
plt.ylabel("$\eta$")
# plt.title("Scaling of BP_reject bound gap with $n$ (normal)")
plt.legend(loc='upper left', ncols=1, frameon=False)
plt.tight_layout()
plt.savefig("eta_concordant.pdf", bbox_inches='tight')

In [ ]:
### the gap experiment
from tqdm import tqdm
ns = [500, 1000, 2000]
ms = [20/1000 + i * 40/1000 for i in range(13)]
# ns = [100]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm]
loss_type = 'concordant'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479

all_results = []
for n in ns:
    for m in ms:
        for dist in dists:
            for seed in tqdm(seeds):
                x = dist.rvs(size=n)
                theta_hat = estimate_theta(x, delta=delta, loss_type=loss_type)
                eta = max(eta_theta_plus(x, theta_hat, int(m*n), delta=delta, loss_type=loss_type), eta_theta_minus(x, theta_hat, int(m*n), delta=delta, loss_type=loss_type))

                all_results.append({
                    'm': m,
                    # 'dist': getattr(dist, 'name', dist.__class__.__name__),
                    'seed': seed,
                    'eta': eta,
                    'n': n,
                })

df = pd.DataFrame(all_results)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# 1) pick your style & context
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

# thresholded manually before 450
df = df[df['m'] < 500/1000]

# 3) build the scatter + regression line
plt.figure(figsize=(8,6))
sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 500],
    color=palette[0],
    label=f'500',
    marker='o',
    linestyle='-',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 1000],
    color=palette[1],
    label=f'1000',
    marker='s',
    linestyle='--',
    alpha=0.8
)

sns.lineplot(
    x='m',
    y='eta',
    data=df[df['n'] == 2000],
    color=palette[2],
    label=f'2000',
    marker='^',
    linestyle=':',
    alpha=0.8
)

# 4) polish labels & legend
plt.xlabel("$m/n$")
plt.ylabel("$\eta$")
# plt.title("Scaling of BP_reject bound gap with $n$ (normal)")
plt.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=3, frameon=False)
plt.tight_layout()
plt.savefig("eta_n_concordant.pdf", bbox_inches='tight')
